In [1]:
%%capture
!pip install facenet-pytorch

## Libraries

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

In [3]:
import numpy as np
import cv2
import os
import time
import torch
import torchvision
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
from torch.utils.data import Subset
from image_iter import FaceDataset, customSubset


from utils import extract_face, take_picture, detect_crop_image, next_folder_name, aug_img, create_neg_class

%matplotlib inline
import matplotlib.pyplot as plt

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

In [ ]:
num_classes = 1
num_images = 50

def create_class_data(data_folder, num_classes, num_images, aug_transform, dataset, idx_dict, model, crop_transform, device):
    class_path = []

    if num_classes == 1:
        path = os.path.join(data_folder, '{}'.format(next_folder_name(data_folder)))
        class_path.append(path)
        try:
            os.mkdir(path)
        except:
            print('Path exists: Issue!')
        create_neg_class(path, num_images, aug_transform, dataset, idx_dict) # add 50 random images

        frame, save_path = extract_face(save_path=data_folder)
        class_path.append(save_path)
        save_path = os.path.join(save_path, '0.jpg')
        images = detect_crop_image(frame=frame, model=mtcnn, transform=crop_transform, device=device)
        aug_img(save_path=save_path, transform=aug_transform, num_images=50)

    else:
        pass
    
    return class_path

## Models

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [5]:
mtcnn = MTCNN(
    image_size=112,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device
)

## Transforms

In [6]:
transform = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
        ])

In [7]:
aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=.5, hue=.3),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)), 
    transforms.RandomAutocontrast(),
    transforms.ToPILImage()])

## Create Dataset

In [8]:
data_path = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data'

In [9]:
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


In [10]:
ss = customSubset(train_root)
idx_dict = ss.generate_idx_dic()

In [15]:
num_classes = 1
num_images = 50

In [16]:
class_paths = create_class_data(data_folder=data_path, 
                                num_classes=num_classes, 
                                num_images=num_images, 
                                aug_transform=aug_transform, 
                                dataset=dataset, 
                                idx_dict=idx_dict, 
                                model=mtcnn, 
                                crop_transform=transform, 
                                device=device)

[ WARN:0@65.998] global /croot/opencv-suite_1691620365762/work/modules/videoio/src/cap_gstreamer.cpp (862) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
libGL error: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: iris
libGL error: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: iris
libGL error: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: swrast


Photo taken!


In [17]:
class_paths

['/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data/0',
 '/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data/1']